# Graph-of-Thought — Colab GPU runner (T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arrafmousa/graph-of-thought/blob/master/notebooks/colab_run.ipynb)

Runs the **reasoning-graph generation POC**: samples token-level reasoning chains for
GSM8K questions with a **frozen** `meta-llama/Llama-3.2-1B-Instruct` (FP16), records
per-token hidden states + log-probabilities, consolidates them into a DAG under a sweep
of latent-merge heuristics/thresholds, and renders a standalone HTML graph report.
Follows `AGENTS.md` §31 — set **Runtime → Change runtime type → T4 GPU** first.

`Llama-3.2` is **gated**: request access on its model page and add a Hugging Face token
as a Colab secret named `HF_TOKEN`. Dataset + model are chosen entirely in
`configs/graph_gsm8k.json` (explicit HF ids/revisions — no hidden defaults). The run writes
`output/<run_id>/` (traces, consolidated graphs, HTML report, manifest, telemetry,
dashboard); the last cell zips it for download.


In [ ]:
# 1) Clone the repo
# Public repo:
!git clone https://github.com/arrafmousa/graph-of-thought.git

# Private repo instead? Store a GitHub token as a Colab secret named GH_TOKEN, then:
# from google.colab import userdata
# tok = userdata.get('GH_TOKEN')
# !git clone https://{tok}@github.com/arrafmousa/graph-of-thought.git

%cd graph-of-thought

In [ ]:
# 2) Install training deps (Colab already ships a CUDA build of torch)
!pip install -q -r requirements.txt
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 3) Authenticate for the gated Llama-3.2 model (needs an HF_TOKEN Colab secret)
from google.colab import userdata
from huggingface_hub import login

login(userdata.get('HF_TOKEN'))  # Colab secret; never committed (AGENTS.md section 8)


In [ ]:
# 4) Generate reasoning graphs on GSM8K: traces -> consolidated DAGs -> HTML report
!python scripts/generate_graphs.py --config configs/graph_gsm8k.json


In [ ]:
# 5) Zip + download the run (Colab disk is ephemeral) and preview the graph report inline
import glob, os, json
from google.colab import files
from IPython.display import HTML

latest = sorted(glob.glob('output/*graph-gsm8k*/'))[-1].rstrip('/')
run_id = os.path.basename(latest)
!zip -qr {run_id}.zip {latest}
files.download(f'{run_id}.zip')

manifest = json.load(open(os.path.join(latest, 'run_manifest.json')))
report = manifest['outputs']['reports'][0]
print('Run:', run_id, '| graph report:', report)
HTML(open(os.path.join(latest, report)).read())


### Optional
- **CPU smoke test (no GPU, no token):** `!python scripts/generate_graphs.py --config configs/graph_demo.json`
- **Validate a downloaded run locally:** `python scripts/validate_run.py output/<run_id>`
- **Sentiment fine-tuning demo (separate workload):** `!python scripts/train.py --config configs/train_sst2.json`
